In [1]:
from langchain_core.documents import Document

In [2]:
doc=Document(
    page_content="this is the main text that will be used for the rag",
    metadata={
        "source":"dummy.txt",
        "pages":10,
        "author":"Sohan",
        
    }
)

In [3]:
doc

Document(metadata={'source': 'dummy.txt', 'pages': 10, 'author': 'Sohan'}, page_content='this is the main text that will be used for the rag')

In [4]:
import os
os.makedirs("../data/text_files",exist_ok=True)

In [5]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}
for filepath,content in sample_texts.items():
    with open(filepath,"w",encoding="utf-8") as f:
        f.write(content)
print("created and written to files")

created and written to files


In [6]:
from langchain_community.document_loaders import TextLoader

/Users/sohan/Desktop/RAG-PRACTICE/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")

In [8]:
document=loader.load()
document

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]

In [9]:
from langchain_community.document_loaders import DirectoryLoader

In [10]:
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    show_progress=True,
    loader_cls=TextLoader
)

In [11]:
documents=dir_loader.load()
documents[0]

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 1932.41it/s]


Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')

In [12]:
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader

In [13]:
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True
)

In [14]:
pdf_docs=dir_loader.load()
pdf_docs

100%|██████████| 4/4 [00:00<00:00, 11.03it/s]


[Document(metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2018-06-08T19:14:34+00:00', 'source': '../data/pdf/GPT.pdf', 'file_path': '../data/pdf/GPT.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2018-06-08T19:14:34+00:00', 'trapped': '', 'modDate': 'D:20180608191434Z', 'creationDate': 'D:20180608191434Z', 'page': 0}, page_content='Improving Language Understanding\nby Generative Pre-Training\nAlec Radford\nOpenAI\nalec@openai.com\nKarthik Narasimhan\nOpenAI\nkarthikn@openai.com\nTim Salimans\nOpenAI\ntim@openai.com\nIlya Sutskever\nOpenAI\nilyasu@openai.com\nAbstract\nNatural language understanding comprises a wide range of diverse tasks such\nas textual entailment, question answering, semantic similarity assessment, and\ndocument classiﬁcation. Although large unlabeled text corpora are abundant,\nlabeled data for learning these speciﬁc tasks is scarce, making it chal

In [15]:
from pathlib import Path

In [16]:
def process_all_pdf_files(pdf_directory):
    all_docs = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"found {len(pdf_files)} files to process")

    for pdf_file in pdf_files:
        try:
            print(f"loading : {pdf_file.name}")
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load() #this returns a list

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_docs.extend(documents)  

        except Exception as e:
            print(f"error : {e}")

    print(f"total number of documents : {len(all_docs)}")
    return all_docs

In [17]:
print(type(documents))

<class 'list'>


In [18]:
all_pdf_documents=process_all_pdf_files("../data")

found 4 files to process
loading : GPT.pdf
loading : IMAGENET.pdf
loading : BERT.pdf
loading : Attention is all you need.pdf
total number of documents : 48


Splitting the text into chunks

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [20]:
def split_documents(documents,chunk_size=400,chunk_overlap=50):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"Content : {split_docs[0].page_content[:200]}")
        print(f"Metadata : {split_docs[0].metadata}")
    return split_docs

In [21]:
chunks=split_documents(all_pdf_documents)
chunks

split 48 documents into 511 chunks
Content : Improving Language Understanding
by Generative Pre-Training
Alec Radford
OpenAI
alec@openai.com
Karthik Narasimhan
OpenAI
karthikn@openai.com
Tim Salimans
OpenAI
tim@openai.com
Ilya Sutskever
OpenAI
i
Metadata : {'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2018-06-08T19:14:34+00:00', 'source': '../data/pdf/GPT.pdf', 'file_path': '../data/pdf/GPT.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2018-06-08T19:14:34+00:00', 'trapped': '', 'modDate': 'D:20180608191434Z', 'creationDate': 'D:20180608191434Z', 'page': 0, 'source_file': 'GPT.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2018-06-08T19:14:34+00:00', 'source': '../data/pdf/GPT.pdf', 'file_path': '../data/pdf/GPT.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2018-06-08T19:14:34+00:00', 'trapped': '', 'modDate': 'D:20180608191434Z', 'creationDate': 'D:20180608191434Z', 'page': 0, 'source_file': 'GPT.pdf', 'file_type': 'pdf'}, page_content='Improving Language Understanding\nby Generative Pre-Training\nAlec Radford\nOpenAI\nalec@openai.com\nKarthik Narasimhan\nOpenAI\nkarthikn@openai.com\nTim Salimans\nOpenAI\ntim@openai.com\nIlya Sutskever\nOpenAI\nilyasu@openai.com\nAbstract\nNatural language understanding comprises a wide range of diverse tasks such\nas textual entailment, question answering, semantic similarity assessment, and'),
 Document(metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creatio

In [22]:
print(type(chunks))
chunks[0]

<class 'list'>


Document(metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2018-06-08T19:14:34+00:00', 'source': '../data/pdf/GPT.pdf', 'file_path': '../data/pdf/GPT.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2018-06-08T19:14:34+00:00', 'trapped': '', 'modDate': 'D:20180608191434Z', 'creationDate': 'D:20180608191434Z', 'page': 0, 'source_file': 'GPT.pdf', 'file_type': 'pdf'}, page_content='Improving Language Understanding\nby Generative Pre-Training\nAlec Radford\nOpenAI\nalec@openai.com\nKarthik Narasimhan\nOpenAI\nkarthikn@openai.com\nTim Salimans\nOpenAI\ntim@openai.com\nIlya Sutskever\nOpenAI\nilyasu@openai.com\nAbstract\nNatural language understanding comprises a wide range of diverse tasks such\nas textual entailment, question answering, semantic similarity assessment, and')

Embedding store and vectorDB

In [23]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
import uuid
from typing import List,Tuple,Any,Dict
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            self.model=SentenceTransformer(self.model_name)
            print(f"model loaded successfully")
        except Exception as e:
            print(f"failed to load model : {self.model_name} : {e}")
            raise
    
    def generate_embeddings(self,texts:List[str])->np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with dimensions : {embeddings.shape}")
        return embeddings
    
embedding_manager=EmbeddingManager()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5516.00it/s]


model loaded successfully


In [25]:
from sentence_transformers import CrossEncoder

class Reranker:
    def __init__(self, model_name="cross-encoder/ms-marco-MiniLM-L-6-v2"):
        self.model = CrossEncoder(model_name)

    def rerank(self, query, docs):
        pairs = [(query, doc['content']) for doc in docs]
        scores = self.model.predict(pairs)

        for doc, score in zip(docs, scores):
            doc['rerank_score'] = float(score)

        return sorted(docs, key=lambda x: x['rerank_score'], reverse=True)

In [26]:
reranker=Reranker()

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 15901.28it/s]


Here we store in the VectorDB

In [27]:
class VectorStore:
    def __init__(self,collection_name:str="pdf_documents",persistent_dir:str="../data/vector_store"):
        self.collection_name=collection_name
        self.persistent_dir=persistent_dir
        self.collection=None
        self.client=None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            os.makedirs(self.persistent_dir,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persistent_dir)

            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"desc":"pdf embeddings for RAG"}
            )
            print(f"vector store initialized :{self.collection_name}")
            print(f"existing documents in the collection : {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store : {e}")
            raise
    
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents must be equal to the number of embeddings")
        
        print(f"adding {len(documents)} documents to the vector store")

        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"added {len(documents)} documents to the vector store")
        
        except Exception as e:
            print(f"error storing the documents into the vector store : {e}")
            raise

vectorstore=VectorStore()

vector store initialized :pdf_documents
existing documents in the collection : 1319


In [28]:
vectorstore

In [29]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2018-06-08T19:14:34+00:00', 'source': '../data/pdf/GPT.pdf', 'file_path': '../data/pdf/GPT.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2018-06-08T19:14:34+00:00', 'trapped': '', 'modDate': 'D:20180608191434Z', 'creationDate': 'D:20180608191434Z', 'page': 0, 'source_file': 'GPT.pdf', 'file_type': 'pdf'}, page_content='Improving Language Understanding\nby Generative Pre-Training\nAlec Radford\nOpenAI\nalec@openai.com\nKarthik Narasimhan\nOpenAI\nkarthikn@openai.com\nTim Salimans\nOpenAI\ntim@openai.com\nIlya Sutskever\nOpenAI\nilyasu@openai.com\nAbstract\nNatural language understanding comprises a wide range of diverse tasks such\nas textual entailment, question answering, semantic similarity assessment, and'),
 Document(metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creatio

In [30]:
texts=[doc.page_content for doc in chunks]
embeddings=embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks,embeddings)

Batches: 100%|██████████| 16/16 [00:02<00:00,  7.18it/s]


Generated embeddings with dimensions : (511, 384)
adding 511 documents to the vector store
added 511 documents to the vector store


Retriever pipeline from the VectorStore

In [31]:
class RAGretriever:
    def __init__(self, vector_store:VectorStore,embedding_manager:EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager

    def retrieve(self,query:str,top_k:int = 20,score_threshold:float = 0.0)->list[Dict[str,Any]]: #the query is the question, the top_k is the top 5 results in this case and the threshold is the minimum similarity threshold, like min they should be like 0 similar
        query_embedding=self.embedding_manager.generate_embeddings([query])[0]

        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_docs=[]

            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]
                seen_contents=set()
                for i,(doc_id,document,metadata,distance) in enumerate (zip(ids,documents,metadatas,distances)):
                    similarity_score=1-distance

                    key=document[:100]
                    if key in seen_contents:
                        continue
                    
                    seen_contents.add(key)
                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                print(f"retrieved {len(retrieved_docs)} documents")
            else:
                print("No documents found")
            return retrieved_docs
        
        except Exception as e:
            print(f"error occurred : {e}")
            return []



rag_retriever=RAGretriever(vectorstore,embedding_manager)


In [32]:
rag_retriever

In [33]:

rag_retriever.retrieve("Unified Multi-Task Learning Framework")

Batches: 100%|██████████| 1/1 [00:00<00:00, 18.09it/s]

Generated embeddings with dimensions : (1, 384)
retrieved 7 documents


[{'id': 'doc_d4be604e_53',
  'content': 'likely our model will beneﬁt from multi-task training as well but we have not explored this currently.\n2https://ftfy.readthedocs.io/en/latest/\n3https://spacy.io/\n5',
  'metadata': {'page': 4,
   'doc_index': 53,
   'creationdate': '2018-06-08T19:14:34+00:00',
   'source_file': 'GPT.pdf',
   'source': '../data/pdf/GPT.pdf',
   'author': '',
   'file_path': '../data/pdf/GPT.pdf',
   'producer': 'pdfTeX-1.40.18',
   'moddate': '2018-06-08T19:14:34+00:00',
   'modDate': 'D:20180608191434Z',
   'trapped': '',
   'title': '',
   'subject': '',
   'format': 'PDF 1.5',
   'creationDate': 'D:20180608191434Z',
   'total_pages': 12,
   'keywords': '',
   'file_type': 'pdf',
   'content_length': 163,
   'creator': 'LaTeX with hyperref package'},
  'similarity_score': 0.21553432941436768,
  'distance': 0.7844656705856323,
  'rank': 1},
 {'id': 'doc_8bdd9ebf_361',
  'content': 'coders.\nIn Proceedings of the 25th international\nconference on Machine learni

In [34]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2018-06-08T19:14:34+00:00', 'source': '../data/pdf/GPT.pdf', 'file_path': '../data/pdf/GPT.pdf', 'total_pages': 12, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2018-06-08T19:14:34+00:00', 'trapped': '', 'modDate': 'D:20180608191434Z', 'creationDate': 'D:20180608191434Z', 'page': 0, 'source_file': 'GPT.pdf', 'file_type': 'pdf'}, page_content='Improving Language Understanding\nby Generative Pre-Training\nAlec Radford\nOpenAI\nalec@openai.com\nKarthik Narasimhan\nOpenAI\nkarthikn@openai.com\nTim Salimans\nOpenAI\ntim@openai.com\nIlya Sutskever\nOpenAI\nilyasu@openai.com\nAbstract\nNatural language understanding comprises a wide range of diverse tasks such\nas textual entailment, question answering, semantic similarity assessment, and\ndocument classiﬁcation. Although large unlabeled text corpora are abundant,\nlabeled data for learning

In [35]:
print(type(all_pdf_documents))
print(type(all_pdf_documents[0]))
print(all_pdf_documents[0])

<class 'list'>
<class 'langchain_core.documents.base.Document'>
page_content='Improving Language Understanding
by Generative Pre-Training
Alec Radford
OpenAI
alec@openai.com
Karthik Narasimhan
OpenAI
karthikn@openai.com
Tim Salimans
OpenAI
tim@openai.com
Ilya Sutskever
OpenAI
ilyasu@openai.com
Abstract
Natural language understanding comprises a wide range of diverse tasks such
as textual entailment, question answering, semantic similarity assessment, and
document classiﬁcation. Although large unlabeled text corpora are abundant,
labeled data for learning these speciﬁc tasks is scarce, making it challenging for
discriminatively trained models to perform adequately. We demonstrate that large
gains on these tasks can be realized by generative pre-training of a language model
on a diverse corpus of unlabeled text, followed by discriminative ﬁne-tuning on each
speciﬁc task. In contrast to previous approaches, we make use of task-aware input
transformations during ﬁne-tuning to achieve effec

Now we integrate the VectorDB context pipeline with the LLM Output

In [36]:
from langchain_community.llms import Ollama

In [37]:
llm=Ollama(model="llama3.2:3b")

/var/folders/mh/zhj0l1nj36g96c4x3vngnn0h0000gn/T/ipykernel_7326/2243293593.py:1: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm=Ollama(model="llama3.2:3b")


In [38]:
def generate_answer(query,retrieved_docs):
    MAX_CHARS = 2000

    context = ""
    for doc in retrieved_docs:
        if len(context) + len(doc['content']) > MAX_CHARS:
            break
        context += doc['content'] + "\n\n"

    prompt = f"""
    You are a technical assistant.

    Answer the question using the context below.
    If the context is incomplete, then just tell me that the context is incomplete and reply with "Not enough context".
    
    If not enough context then abruptly reply with "Not enough context", don
    t try to fit in text yourself.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """
    return llm.invoke(prompt)

In [39]:
retrieved=rag_retriever.retrieve("transformer self attention mechanism explanation")
answer=generate_answer("what is a transformer used for?",retrieved)
print(answer)

Batches: 100%|██████████| 1/1 [00:00<00:00, 138.43it/s]

Generated embeddings with dimensions : (1, 384)
retrieved 4 documents


A Transformer model is primarily used for natural language processing tasks such as:

1. Language modeling: predicting the next word in a sequence.
2. Machine translation: translating text from one language to another.

In addition, it can be applied to other tasks like question answering, sentiment analysis, and text summarization.


In [40]:
rag_retriever=RAGretriever(vectorstore,embedding_manager)

advanced rag pipeline

In [41]:
def adv_rag(query,rag_retriever,llm,top_k=5,min_score=0.01,return_context=False):
    results=rag_retriever.retrieve(query,top_k,score_threshold=min_score)
    if not results:
        return {'answer': 'no relevant context found','sources':[],'confidence':0.0}
    
    context='\n\n'.join([doc['content'] for doc in results])
    sources=[{
        'source':doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
        'page':doc['metadata'].get('page','unknown'),
        'score':doc['similarity_score'],
        'preview':doc['content'][:150]+'...'
    } for doc in results]
    
    confidence=max(doc['similarity_score'] for doc in results)
    
    prompt=f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response=llm.invoke(prompt)
    output={
        'answer':response,
        'sources':sources,
        'confidence':confidence
    }
    if return_context:
        output['context']=context
    return output
    

In [42]:
results = rag_retriever.retrieve(query="transformer",top_k=20)
reranked=reranker.rerank("transformer",results)
results=reranked[:5]
print(results)

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.59it/s]


Generated embeddings with dimensions : (1, 384)
retrieved 4 documents
[{'id': 'doc_3347915c_72', 'content': '88.1\n56.0\nTransformer w/o pre-training\n59.9\n18.9\n84.0\n79.4\n30.9\n65.5\n75.7\n71.2\n53.8\nTransformer w/o aux LM\n75.0\n47.9\n92.0\n84.9\n83.2\n69.8\n81.1\n86.9\n54.4\nLSTM w/ aux LM\n69.1\n30.3\n90.5\n83.2\n71.8\n68.1\n73.7\n81.1\n54.6\nattentional memory of the transformer assists in transfer compared to LSTMs. We designed a series', 'metadata': {'keywords': '', 'moddate': '2018-06-08T19:14:34+00:00', 'author': '', 'total_pages': 12, 'creationDate': 'D:20180608191434Z', 'source': '../data/pdf/GPT.pdf', 'creationdate': '2018-06-08T19:14:34+00:00', 'page': 7, 'trapped': '', 'subject': '', 'modDate': 'D:20180608191434Z', 'format': 'PDF 1.5', 'file_type': 'pdf', 'creator': 'LaTeX with hyperref package', 'doc_index': 72, 'content_length': 309, 'source_file': 'GPT.pdf', 'title': '', 'file_path': '../data/pdf/GPT.pdf', 'producer': 'pdfTeX-1.40.18'}, 'similarity_score': 0.064074

In [43]:
result=adv_rag("what is transformer?",rag_retriever=rag_retriever,llm=llm,return_context=True)
print('answer:',result['answer'])
print('sources:',result['sources'])
print('confidence:',result['confidence'])
# print('context:',result['context'][:150])

Batches: 100%|██████████| 1/1 [00:00<00:00, 39.29it/s]

Generated embeddings with dimensions : (1, 384)
retrieved 1 documents


answer: The Transformer is a neural network architecture that relies entirely on self-attention for computing representations of its input and output, without using sequence-aligned RNNs or convolution.
sources: [{'source': 'Attention is all you need.pdf', 'page': 1, 'score': 0.02252805233001709, 'preview': 'language modeling tasks [28].\nTo the best of our knowledge, however, the Transformer is the ﬁrst transduction model relying\nentirely on self-attention...'}]
confidence: 0.02252805233001709


In [50]:
from typing import List, Dict, Any
import time
class AdvRag:
    def __init__(self,retriever,llm):
        self.retriever=retriever
        self.llm=llm
        self.history=[]
    

    def query(self,question:str,top_k:int=20,min_score:float=0.01,summarize:bool=False)->Dict[str,any]:
        results=self.retriever.retrieve(question,top_k=top_k,score_threshold=min_score)
        reranked=reranker.rerank(question,results)
        results=reranked[:5]
        seen_sources = set()
        filtered = []

        for doc in reranked:
            source = doc['metadata'].get('source_file')

            if source not in seen_sources:
                filtered.append(doc)
                seen_sources.add(source)

            if len(filtered) == 5:
                break

        results = filtered
        if not results:
            return {
                'answer':"no relevant documents found",
                'sources':[],
                'context':""
            }
        else:
            context='\n\n'.join(doc['content'] for doc in results)
            sources=[{
        'source':doc['metadata'].get('source_file',doc['metadata'].get('source','unknown')),
        'page':doc['metadata'].get('page','unknown'),
        'score':doc['similarity_score'],
        'preview':doc['content'][:150]+'...'
        } for doc in results]
        
        prompt=f"""use the following context to answer the question concisely, context{context}, question: {question}\n\nAnswer:"""
        response=self.llm.invoke([prompt.format(context=context,question=question)])
        answer=response
        summary=None
        if summarize and answer:
            summary_prompt=f"summarize the following answer in like 2 sentences {answer}"
            summary_resp=self.llm.invoke([summary_prompt])
            summary=summary_resp

        self.history.append({
            'question':question,
            'answer':answer,
            'sources':sources,
            'summary':summary
        })
        return {
            'question':question,
            'answer':answer,
            'sources':sources,
            'summary':summary,
            'history':self.history
        }

In [51]:
advanced_rag=AdvRag(rag_retriever,llm)
result=advanced_rag.query("what is transformer",summarize=True)
print('answer:',result['answer'])
print('Summary:',result['summary'])
print("Hoistory:",result['history'][-1])
result=advanced_rag.query("what is attention mechanism",summarize=True)
print('answer:',result['answer'])
print('Summary:',result['summary'])
print("Hoistory:",result['history'][-2])

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.49it/s]


Generated embeddings with dimensions : (1, 384)
retrieved 1 documents
answer: The Transformer is a model architecture that uses self-attention mechanisms and consists of two main sub-layers: an encoder layer with a multi-head fully connected feed-forward network.
Summary: Here's a summary:

The Transformer is a neural network architecture using self-attention, consisting of two main layers: the encoder layer and a multi-head fully connected feed-forward network. The architecture is designed to handle sequential data efficiently, making it well-suited for natural language processing tasks.
Hoistory: {'question': 'what is transformer', 'answer': 'The Transformer is a model architecture that uses self-attention mechanisms and consists of two main sub-layers: an encoder layer with a multi-head fully connected feed-forward network.', 'sources': [{'source': 'Attention is all you need.pdf', 'page': 2, 'score': 0.0122528076171875, 'preview': 'Figure 1: The Transformer - model architecture.\nwi

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.10it/s]


Generated embeddings with dimensions : (1, 384)
retrieved 3 documents
answer: Attention mechanism is a way for the model to focus on specific parts of the input data that are relevant for making predictions or taking actions. In the context of self-attention, it's used to weigh the importance of different positions within a single sequence and compute a representation of the sequence by considering all possible relationships between them.
Summary: Here is a 2-sentence summary:

The attention mechanism helps the model focus on relevant parts of the input data for prediction or action. In self-attention, it weighs the importance of different positions within a sequence to create a representation that considers all possible relationships between them.
Hoistory: {'question': 'what is transformer', 'answer': 'The Transformer is a model architecture that uses self-attention mechanisms and consists of two main sub-layers: an encoder layer with a multi-head fully connected feed-forward network

In [52]:
test_queries = [
    "BERT vs GPT",
    "Transformer vs CNN",
    "attention mechanism",
    "how transformers are used in vision",
    "limitations of transformers"
]

In [53]:
for queries in test_queries:
    result=advanced_rag.query(queries,summarize=True)
    print('answer:',result['answer'])
    print('Summary:',result['summary'])
    print("History:",result['history'][-1])
    print("source",result['sources'])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.53it/s]


Generated embeddings with dimensions : (1, 384)
retrieved 7 documents
answer: BERT and OpenAI GPT are comparable in purpose (pre-training language models), but differ in approach: BERT is a feature-based approach using fine-tuning, while GPT uses a left-to-right Transformer LM with pre-trained parameters.
Summary: Here is a summary of the answer in 2 sentences:

BERT and OpenAI GPT are comparable language models that aim to improve understanding of language through pre-training. While BERT takes a feature-based approach using fine-tuning, GPT uses a different architecture based on left-to-right Transformers with pre-trained parameters.
History: {'question': 'BERT vs GPT', 'answer': 'BERT and OpenAI GPT are comparable in purpose (pre-training language models), but differ in approach: BERT is a feature-based approach using fine-tuning, while GPT uses a left-to-right Transformer LM with pre-trained parameters.', 'sources': [{'source': 'BERT.pdf', 'page': 13, 'score': 0.4070241451263428, '

Batches: 100%|██████████| 1/1 [00:00<00:00,  7.23it/s]

Generated embeddings with dimensions : (1, 384)
retrieved 4 documents


...

Batches: 100%|██████████| 1/1 [00:00<00:00,  8.48it/s]

Generated embeddings with dimensions : (1, 384)
retrieved 1 documents


answer: Transformers are increasingly being used in computer vision tasks, particularly those requiring long-range dependencies, such as object detection and segmentation. They offer advantages over traditional CNN-based approaches by allowing for parallelization of attention mechanisms, reducing computational complexity, and improving modeling capacity, especially when combined with Multi-Head Attention to counteract positional encoding limitations.
Summary: Transformers are being increasingly used in computer vision tasks like object detection and segmentation due to their ability to handle long-range dependencies and improve modeling capacity. They offer advantages over traditional CNN-based approaches by allowing for parallelization of attention mechanisms, reducing computational complexity, and improving performance when combined with Multi-Head Attention.
History: {'question': 'how transformers are used in vision', 'answer': 'Transformers are increasingly being used in computer v

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.77it/s]


Generated embeddings with dimensions : (1, 384)
retrieved 2 documents
answer: Here is a concise answer:

The limitations of Transformers include:

1. **Sequential generation**: Current models struggle to generate output sequences that are not sequential in nature, such as generating images or music.
2. **Large input and output sizes**: Traditional Transformer architecture can become computationally expensive with large inputs and outputs, requiring significant memory and computational resources.
3. **Limited handling of modalities other than text**: Existing Transformers require modifications to accommodate input and output modalities like images, audio, and video.

These limitations highlight the need for further research and development in extending the Transformer architecture to address these challenges.
Summary: Here is a 2-sentence summary:

Transformers have limitations that hinder their ability to generate non-sequential content, require significant computational resources with